In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported ✅")

Libraries imported ✅


In [3]:
df = pd.read_csv("C:/Users/Yashwanth u/Desktop/Data_projects/ab-testing/data/raw/ab_data.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst look:")
df.head(10)

Shape: (294478, 5)

Columns: ['user_id', 'timestamp', 'group', 'landing_page', 'converted']

First look:


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1
5,936923,2017-01-10 15:20:49.083499,control,old_page,0
6,679687,2017-01-19 03:26:46.940749,treatment,new_page,1
7,719014,2017-01-17 01:48:29.539573,control,old_page,0
8,817355,2017-01-04 17:58:08.979471,treatment,new_page,1
9,839785,2017-01-15 18:11:06.610965,treatment,new_page,1


In [4]:
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nNull Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (294478, 5)

Data Types:
 user_id          int64
timestamp       object
group           object
landing_page    object
converted        int64
dtype: object

Null Values:
 user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

Duplicate Rows: 0


In [5]:
print("Unique Groups:", df['group'].unique())
print("Unique Landing Pages:", df['landing_page'].unique())
print("Unique Converted Values:", df['converted'].unique())
print("\nGroup Distribution:\n", df['group'].value_counts())
print("\nLanding Page Distribution:\n", df['landing_page'].value_counts())
print("\nConversion Distribution:\n", df['converted'].value_counts())

Unique Groups: ['control' 'treatment']
Unique Landing Pages: ['old_page' 'new_page']
Unique Converted Values: [0 1]

Group Distribution:
 group
treatment    147276
control      147202
Name: count, dtype: int64

Landing Page Distribution:
 landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64

Conversion Distribution:
 converted
0    259241
1     35237
Name: count, dtype: int64


In [6]:
# Control group should see old page
# Treatment group should see new page
# Any mismatch = dirty data

mismatch = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
]

print("Mismatched rows:", len(mismatch))
print("\nSample mismatches:")
mismatch.head()

Mismatched rows: 3893

Sample mismatches:


,user_id,timestamp,group,landing_page,converted
22,767017,2017-01-12 22:58:14.991443,control,new_page,0
240,733976,2017-01-11 15:11:16.407599,control,new_page,0
308,857184,2017-01-20 07:34:59.832626,treatment,old_page,0
327,686623,2017-01-09 14:26:40.734775,treatment,old_page,0
357,856078,2017-01-12 12:29:30.354835,treatment,old_page,0


In [7]:
before = df.shape[0]

# Keep only correct combinations
df = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
]

print(f"Removed {before - df.shape[0]} mismatched rows")
print("Shape now:", df.shape)

Removed 3893 mismatched rows
Shape now: (290585, 5)


In [8]:
# Same user should not appear twice
duplicate_users = df[df.duplicated(subset=['user_id'])]
print("Duplicate user IDs:", len(duplicate_users))

# Keep only first occurrence per user
df = df.drop_duplicates(subset=['user_id'], keep='first')
print("After removing duplicates:", df.shape)

Duplicate user IDs: 1
After removing duplicates: (290584, 5)


In [9]:
# Fix timestamp column
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Extract time features
df['date']     = df['timestamp'].dt.date
df['hour']     = df['timestamp'].dt.hour
df['day']      = df['timestamp'].dt.day_name()

print("Data types fixed ✅")
print(df.dtypes)

Data types fixed ✅
user_id                  int64
timestamp       datetime64[ns]
group                   object
landing_page            object
converted                int64
date                    object
hour                     int32
day                     object
dtype: object


In [10]:
print("=" * 50)
print("FINAL CLEANED DATA SUMMARY")
print("=" * 50)
print(f"Total Rows:          {df.shape[0]:,}")
print(f"Total Columns:       {df.shape[1]}")
print(f"Null Values:         {df.isnull().sum().sum()}")
print(f"Duplicate Users:     {df.duplicated(subset=['user_id']).sum()}")
print(f"Control Group:       {len(df[df['group']=='control']):,}")
print(f"Treatment Group:     {len(df[df['group']=='treatment']):,}")
print(f"Total Conversions:   {df['converted'].sum():,}")
print(f"Overall Conv Rate:   {df['converted'].mean()*100:.2f}%")
print("=" * 50)

FINAL CLEANED DATA SUMMARY
Total Rows:          290,584
Total Columns:       8
Null Values:         0
Duplicate Users:     0
Control Group:       145,274
Treatment Group:     145,310
Total Conversions:   34,753
Overall Conv Rate:   11.96%


In [11]:
df.to_csv('C:/Users/Yashwanth u/Desktop/Data_projects/ab-testing/data/raw/ab_data_clean.csv', index=False)
print("✅ Cleaned data saved!")

✅ Cleaned data saved!
